# Imports

In [2]:
#!pip install -q pandas rapidfuzz python-dateutil scikit-learn openpyxl

import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein
from datetime import timedelta
from dateutil import parser
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Cleanup functions

In [4]:
def parse_date(x):
    if pd.isna(x) or x in ("", "N/A", "NA", None):
        return None
    s = str(x)
    try:
        return parser.parse(s, fuzzy=True, dayfirst=False).date()
    except Exception:
        return None

def normalize_name(s):
    if pd.isna(s) or not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = s.replace("'", " ").replace("-", " ")
    s = "".join(ch if ch.isalnum() or ch.isspace() else " " for ch in s)
    return " ".join(s.split())

def normalize_digits(x):
    if pd.isna(x) or x in ("", None):
        return ""
    s = str(x)
    return "".join(ch for ch in s if ch.isalnum()).lower()

def is_valid_ssn(x):
    d = normalize_digits(x)
    if len(d) != 9 or not d.isdigit():
        return False
    if d in {"000000000","111111111","999999999"}:
        return False
    if d[:3] == "000" or d[3:5] == "00" or d[5:] == "0000":
        return False
    if len(set(d)) == 1:
        return False
    return True


## Load the data

In [6]:
DB_PATH = "synthetic_patient_db_200k_v16_hardneg80k.csv"
IN_PATH = "incoming_records_50k_v16_all_true_hardneg80k.csv"
TRUTH_PATH = "ground_truth_links_v16_all_true.csv"

db = pd.read_csv(DB_PATH)
inc = pd.read_csv(IN_PATH)
truth = pd.read_csv(TRUTH_PATH)

db.shape, inc.shape, truth.shape

((200000, 8), (50000, 8), (50000, 3))

## Normalize DB and Incoming
Create standardized columns used for exact joins.
- `dob_dt`, `doi_dt`: parsed dates
- `ssn_norm`, `claim_norm`: alphanumeric-only
- `name_norm`: normalized name
- `ssn_valid`: valid SSN flag


In [8]:
dbn = db.copy()
incn = inc.copy()

dbn["dob_dt"] = dbn["patient_dob"].apply(parse_date)
dbn["doi_dt"] = dbn["patient_doi"].apply(parse_date)
dbn["ssn_norm"] = dbn["patient_ssn"].apply(normalize_digits)
dbn["ssn_valid"] = dbn["patient_ssn"].apply(is_valid_ssn)
dbn["name_norm"] = dbn["patient_name"].apply(normalize_name)
dbn["claim_norm"] = dbn["claim_id"].apply(normalize_digits)

incn["dob_dt"] = incn["patient_dob"].apply(parse_date)
incn["doi_dt"] = incn["patient_doi"].apply(parse_date)
incn["ssn_norm"] = incn["patient_ssn"].apply(normalize_digits)
incn["ssn_valid"] = incn["patient_ssn"].apply(is_valid_ssn)
incn["name_norm"] = incn["patient_name"].apply(normalize_name)
incn["claim_norm"] = incn["claim_id"].apply(normalize_digits)

dbn[["dob_dt","doi_dt","ssn_valid"]].isna().mean(), incn[["dob_dt","doi_dt","ssn_valid"]].isna().mean()

(dob_dt       0.001740
 doi_dt       0.001635
 ssn_valid    0.000000
 dtype: float64,
 dob_dt       0.00920
 doi_dt       0.01004
 ssn_valid    0.00000
 dtype: float64)

# CURRENT BUSINESS LOGIC

## What is `m1`?
`m1` is the candidate match table from **Lookup 1A (SSN path)**.

### Lookup 1A logic
1) Filter incoming to rows where SSN is valid and DOB/DOI exist.
2) Filter DB to rows where SSN is valid and DOB/DOI exist.
3) **Join** on exact keys: `(indexor_code, ssn_norm, dob_dt)`.
4) Compute `doi_diff` and keep only `doi_diff <= 7`.
5) If multiple DB candidates exist, keep the “best” (smallest doi_diff, then smallest db_record_id).


In [11]:
# Lookup 1A (SSN) -> m1
inc1 = incn[incn["ssn_valid"] & incn["dob_dt"].notna() & incn["doi_dt"].notna()].copy()
db1  = dbn[dbn["ssn_valid"] & dbn["dob_dt"].notna() & dbn["doi_dt"].notna()].copy()

m1 = inc1.merge(
    db1[["db_record_id","indexor_code","ssn_norm","dob_dt","doi_dt"]],
    on=["indexor_code","ssn_norm","dob_dt"],
    how="left",
    suffixes=("_inc","_db")
)

# keep successful joins
m1 = m1[m1["db_record_id"].notna()].copy()

# DOI window filter
m1["doi_diff"] = (pd.to_datetime(m1["doi_dt_inc"]) - pd.to_datetime(m1["doi_dt_db"])).dt.days.abs()
m1 = m1[m1["doi_diff"] <= 7]

# select best match per incoming record
m1 = m1.sort_values(["incoming_record_id","doi_diff","db_record_id"]).drop_duplicates("incoming_record_id")

m1.head(), m1.shape

(       incoming_record_id         patient_name        patient_dob  \
 11237                   1      Mark x greeN jr   October 09, 2000   
 17362                   7        kathleen wood  November 25, 1951   
 8160                    8      AMANDA F III JR         10-19-1948   
 20072                  10   charles c. hill sr         1942-06-26   
 3386                   14  STEPHEN CAMPBELL SR         05/07/1994   
 
              patient_doi        patient_dos  patient_ssn  claim_id  \
 11237         2024/04/15         2024/04/23  684 53 2260  Y-03t267   
 17362         04/23/2025         05/23/2025  876 47 8323   J365805   
 8160          08/24/2023        Aug 28 2023  965 79 1909   B453M50   
 20072         2016/10/16        11/11/2016   497-95-8840  vu374651   
 3386   February 10, 2024  February 27, 2024  792-84-6929   z923235   
 
       indexor_code      dob_dt  doi_dt_inc   ssn_norm  ssn_valid  \
 11237          AXA  2000-10-09  2024-04-15  684532260       True   
 17362      

## How `m1` updates `pred`
`pred` is the running output. After Lookup 1A, fill `pred_db_record_id` and set `match_method='L1_SSN'`.

In [13]:
pred = pd.DataFrame({"incoming_record_id": incn["incoming_record_id"].values})
pred["pred_db_record_id"] = np.nan
pred["match_method"] = ""

pred = pred.merge(m1[["incoming_record_id","db_record_id"]], on="incoming_record_id", how="left")
pred["pred_db_record_id"] = pred["db_record_id"]
pred["match_method"] = np.where(pred["pred_db_record_id"].notna(), "L1_SSN", pred["match_method"])
pred = pred.drop(columns=["db_record_id"])

pred.head()

,incoming_record_id,pred_db_record_id,match_method
0,37978,23807.0,L1_SSN
1,20385,NaN,
2,1595,NaN,
3,46711,176039.0,L1_SSN
4,17984,NaN,


## Lookup 1B (Name path) -> `m2`
Only run on rows still unmatched after SSN path.
Join keys: `(indexor_code, name_norm, dob_dt)` then DOI window filter.

In [15]:
remaining = pred[pred["pred_db_record_id"].isna()]["incoming_record_id"]
inc_rem = incn[incn["incoming_record_id"].isin(remaining)].copy()

inc2 = inc_rem[(~inc_rem["ssn_valid"]) & inc_rem["dob_dt"].notna() & inc_rem["doi_dt"].notna()].copy()
db2  = dbn[dbn["dob_dt"].notna() & dbn["doi_dt"].notna()].copy()

m2 = inc2.merge(
    db2[["db_record_id","indexor_code","name_norm","dob_dt","doi_dt"]],
    on=["indexor_code","name_norm","dob_dt"],
    how="left",
    suffixes=("_inc","_db")
)
m2 = m2[m2["db_record_id"].notna()].copy()
m2["doi_diff"] = (pd.to_datetime(m2["doi_dt_inc"]) - pd.to_datetime(m2["doi_dt_db"])).dt.days.abs()
m2 = m2[m2["doi_diff"] <= 7]
m2 = m2.sort_values(["incoming_record_id","doi_diff","db_record_id"]).drop_duplicates("incoming_record_id")

# fill pred
pred = pred.merge(m2[["incoming_record_id","db_record_id"]], on="incoming_record_id", how="left")
pred["pred_db_record_id"] = pred["pred_db_record_id"].fillna(pred["db_record_id"])
pred["match_method"] = np.where((pred["match_method"]=="") & pred["pred_db_record_id"].notna(), "L1_NAME", pred["match_method"])
pred = pred.drop(columns=["db_record_id"])

pred["match_method"].value_counts().head(10)

match_method
L1_SSN     22098
L1_NAME    16406
           11496
Name: count, dtype: int64

## Lookup 2 (Claim path) -> `m3`
Only run on rows still unmatched after Lookup 1.
Join keys: `(indexor_code, claim_norm, dob_dt)` then DOI window filter.

In [17]:
remaining = pred[pred["pred_db_record_id"].isna()]["incoming_record_id"]
inc_rem = incn[incn["incoming_record_id"].isin(remaining)].copy()

inc3 = inc_rem[(inc_rem["claim_norm"]!="") & inc_rem["dob_dt"].notna() & inc_rem["doi_dt"].notna()].copy()
db3  = dbn[(dbn["claim_norm"]!="") & dbn["dob_dt"].notna() & dbn["doi_dt"].notna()].copy()

m3 = inc3.merge(
    db3[["db_record_id","indexor_code","claim_norm","dob_dt","doi_dt"]],
    on=["indexor_code","claim_norm","dob_dt"],
    how="left",
    suffixes=("_inc","_db")
)
m3 = m3[m3["db_record_id"].notna()].copy()
m3["doi_diff"] = (pd.to_datetime(m3["doi_dt_inc"]) - pd.to_datetime(m3["doi_dt_db"])).dt.days.abs()
m3 = m3[m3["doi_diff"] <= 7]
m3 = m3.sort_values(["incoming_record_id","doi_diff","db_record_id"]).drop_duplicates("incoming_record_id")

# fill pred
pred = pred.merge(m3[["incoming_record_id","db_record_id"]], on="incoming_record_id", how="left")
pred["pred_db_record_id"] = pred["pred_db_record_id"].fillna(pred["db_record_id"])
pred["match_method"] = np.where((pred["match_method"]=="") & pred["pred_db_record_id"].notna(), "L2_CLAIM", pred["match_method"])
pred = pred.drop(columns=["db_record_id"])

pred["match_found"] = pred["pred_db_record_id"].notna().astype(int)
pred["match_method"].value_counts().head(10)

match_method
L1_SSN      22098
L1_NAME     16406
             9371
L2_CLAIM     2125
Name: count, dtype: int64

## Evaluate
Recall on true matches = fraction where predicted DB id equals the ground truth DB id.
False-found rate = fraction of true no-match incoming rows where something was still found.

In [19]:
eval_df = pred.merge(truth, on="incoming_record_id", how="left")

true = eval_df[eval_df["is_true_match"]==1].copy()
true["pred_db_int"] = pd.to_numeric(true["pred_db_record_id"], errors="coerce")
true["truth_db_int"] = pd.to_numeric(true["db_record_id"], errors="coerce")
true["correct_match"] = ((true["pred_db_int"].notna()) & (true["pred_db_int"] == true["truth_db_int"])).astype(int)

recall = float(true["correct_match"].mean())
false_found = float(eval_df[eval_df["is_true_match"]==0]["match_found"].mean())

recall, false_found

(0.81252, nan)

# Blocking to generate pairs for training efficiently

In [21]:
def ssn_digits(x):
    if x is None or pd.isna(x): return ""
    return re.sub(r"\D", "", str(x))

def parse_date_robust_series(s: pd.Series) -> pd.Series:
    ss = s.fillna("").astype(str).str.strip()
    dt = pd.to_datetime(ss, errors="coerce")
    mask = dt.isna() & ss.ne("") & ss.ne("nan") & ss.ne("NaN")
    if mask.any():
        out = []
        for v in ss[mask].values:
            try:
                out.append(parser.parse(v, fuzzy=True).date())
            except Exception:
                out.append(pd.NaT)
        dt.loc[mask] = pd.to_datetime(out, errors="coerce")
    return dt.dt.date

def clean_name(s: pd.Series) -> pd.Series:
    return (s.fillna("").astype(str).str.upper().str.strip()
              .str.replace(r"[^A-Z0-9 ]", " ", regex=True)
              .str.replace(r"\s+", " ", regex=True).str.strip())

def last_name_token(clean_full_name: pd.Series) -> pd.Series:
    # Handles "LAST, FIRST" and "FIRST LAST"
    s = clean_full_name.fillna("")
    # if comma, take left side; else take last token
    has_comma = s.str.contains(",", regex=False)
    left = s.where(~has_comma, s.str.split(",").str[0].str.strip())
    tok = left.where(has_comma, s.str.split().str[-1].fillna(""))
    return tok.fillna("")

def add_block_cols(df: pd.DataFrame, prefix: str):
    out = df.copy()
    out["dob_dt"] = parse_date_robust_series(out["patient_dob"])
    out["doi_dt"] = parse_date_robust_series(out["patient_doi"])
    out["doi_month"] = pd.to_datetime(out["doi_dt"], errors="coerce").dt.to_period("M").astype(str)
    out["ssn_last4"] = out["patient_ssn"].apply(ssn_digits).str[-4:].fillna("")
    out["name_clean"] = clean_name(out["patient_name"])
    out["last_name"] = last_name_token(out["name_clean"])
    return out

db_b = add_block_cols(db, "db")
inc_b = add_block_cols(inc, "inc")

# Build four candidate sets via merges (UNION them)
cand_parts = []

# 1) indexor + DOB
cand_dob = inc_b[inc_b["dob_dt"].notna()].merge(
    db_b[["db_record_id","indexor_code","dob_dt"]],
    on=["indexor_code","dob_dt"], how="left"
)[["incoming_record_id","db_record_id"]].dropna()
cand_parts.append(cand_dob)

# 2) indexor + DOI month
cand_doi = inc_b[inc_b["doi_dt"].notna()].merge(
    db_b[["db_record_id","indexor_code","doi_month"]],
    on=["indexor_code","doi_month"], how="left"
)[["incoming_record_id","db_record_id"]].dropna()
cand_parts.append(cand_doi)

# 3) indexor + SSN last4 (only if last4 present)
cand_last4 = inc_b[inc_b["ssn_last4"].str.len() == 4].merge(
    db_b[["db_record_id","indexor_code","ssn_last4"]],
    on=["indexor_code","ssn_last4"], how="left"
)[["incoming_record_id","db_record_id"]].dropna()
cand_parts.append(cand_last4)

# 4) indexor + last name token (only if last_name present)
cand_ln = inc_b[inc_b["last_name"].ne("")].merge(
    db_b[["db_record_id","indexor_code","last_name"]],
    on=["indexor_code","last_name"], how="left"
)[["incoming_record_id","db_record_id"]].dropna()
cand_parts.append(cand_ln)

# UNION + de-dupe
candidates = pd.concat(cand_parts, ignore_index=True).drop_duplicates()

# Cap candidates per incoming to top N by cheap DOI distance
N = 200
tmp = candidates.merge(
    inc_b[["incoming_record_id","doi_dt"]],
    on="incoming_record_id", how="left"
).merge(
    db_b[["db_record_id","doi_dt"]],
    on="db_record_id", how="left",
    suffixes=("_inc","_db")
)
tmp["doi_diff_days"] = (
    pd.to_datetime(tmp["doi_dt_inc"]) - pd.to_datetime(tmp["doi_dt_db"])
).dt.days.abs()

tmp = tmp.sort_values(["incoming_record_id","doi_diff_days","db_record_id"])
candidates_pruned = tmp.groupby("incoming_record_id").head(N)[["incoming_record_id","db_record_id"]]

print("Avg candidates:", candidates_pruned.groupby("incoming_record_id").size().mean())
print("P95 candidates:", candidates_pruned.groupby("incoming_record_id").size().quantile(0.95))

C:\Users\aryan\AppData\Local\Temp\ipykernel_170988\3582061316.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(ss, errors="coerce")


Avg candidates: 155.9073
P95 candidates: 200.0


## Candidate Recall (when blocking, what proportion of the time does the candidate show up)

In [23]:
# --- Candidate Recall ---
# candidates_pruned: incoming_record_id, db_record_id
# ground_truth: incoming_record_id, db_record_id, is_true_match

gt_true = truth[truth["is_true_match"] == 1].copy()

# Merge truth against candidates to see if the true db_record_id is present
hit = gt_true.merge(
    candidates_pruned,
    on=["incoming_record_id", "db_record_id"],
    how="left",
    indicator=True
)

hit["in_candidates"] = (hit["_merge"] == "both").astype(int)

candidate_recall = hit["in_candidates"].mean()

print(f"Candidate recall: {candidate_recall:.4f}")
print(f"Missed true matches: {int((hit['in_candidates'] == 0).sum())} / {len(hit)}")

Candidate recall: 0.9951
Missed true matches: 246 / 50000


## Generate pairwise training dataset as a first step for training ML model

In [25]:

# ============================================================
# Generate Pairwise Training Dataset from Candidates
# ============================================================
# This section builds a "pairs-style" dataset (left_* vs right_*) for ML training.
# Each row is (incoming_record_id, candidate_db_record_id) with label:
#   is_same_person = 1 if candidate_db_record_id == ground truth db_record_id else 0
#
# IMPORTANT:
# - Split train/test by incoming_record_id to avoid leakage.
# - Use 1 positive + K hard negatives per incoming record (sampled from candidate list).
# - Prune candidates per incoming to TOP_N by DOI distance.

import numpy as np
import pandas as pd

# ---- Tunables ----
TOP_N_CANDIDATES = 200      # Prune candidates per incoming after union (set None to disable)
NEGATIVES_PER_POS = 50      # Hard negatives per incoming (from candidate set)
TRAIN_FRAC = 0.80
RANDOM_SEED = 11

# ---- Expected dataframes ----
# db, incoming, ground_truth (already loaded above in this notebook)
# candidates_pruned (from the blocking step) OR candidates (pre-prune) + inc_b/db_b with doi_dt.

assert "db" in globals(), "Expected dataframe `db` to be loaded."
assert "inc" in globals(), "Expected dataframe `incoming` to be loaded."
assert "truth" in globals(), "Expected dataframe `ground_truth` to be loaded."

# If `candidates_pruned` exists already, then use it.
# Otherwise, if `candidates` and the DOI-parsed helper frames (`inc_b`, `db_b`) exist already,
# they'll be pruned here.
if "candidates_pruned" not in globals():
    assert "candidates" in globals(), "Expected `candidates` to exist if `candidates_pruned` is not present."
    assert "inc_b" in globals() and "db_b" in globals(), "Expected `inc_b` and `db_b` for DOI-based pruning."
    tmp = candidates.merge(
        inc_b[["incoming_record_id","doi_dt"]],
        on="incoming_record_id", how="left"
    ).merge(
        db_b[["db_record_id","doi_dt"]],
        on="db_record_id", how="left",
        suffixes=("_inc","_db")
    )
    tmp["doi_diff_days"] = (
        pd.to_datetime(tmp["doi_dt_inc"]) - pd.to_datetime(tmp["doi_dt_db"])
    ).dt.days.abs()

    tmp = tmp.sort_values(["incoming_record_id","doi_diff_days","db_record_id"])
    if TOP_N_CANDIDATES is None:
        candidates_pruned = tmp[["incoming_record_id","db_record_id"]].drop_duplicates()
    else:
        candidates_pruned = (
            tmp.groupby("incoming_record_id")
               .head(TOP_N_CANDIDATES)[["incoming_record_id","db_record_id"]]
               .drop_duplicates()
        )

# ---- Build 1 pos + K neg pairs per incoming ----
gt_true = truth[truth["is_true_match"] == 1].copy()
truth_map = gt_true.set_index("incoming_record_id")["db_record_id"].to_dict()

rng = np.random.default_rng(RANDOM_SEED)

grouped = candidates_pruned.groupby("incoming_record_id")["db_record_id"].apply(list)

pairs_rows = []
missed_pos = 0

for inc_id, cand_list in grouped.items():
    true_db = truth_map.get(inc_id, None)
    if true_db is None or pd.isna(true_db):
        continue

    # skip the few cases where the true match isn't in candidates (candidate-recall misses)
    if true_db not in cand_list:
        missed_pos += 1
        continue

    # positive
    pairs_rows.append((inc_id, true_db, 1))

    # hard negatives from candidate pool
    negs = [c for c in cand_list if c != true_db]
    if not negs:
        continue

    if len(negs) <= NEGATIVES_PER_POS:
        sampled = negs
    else:
        sampled = rng.choice(negs, size=NEGATIVES_PER_POS, replace=False).tolist()

    for db_id in sampled:
        pairs_rows.append((inc_id, db_id, 0))

pairs_index = pd.DataFrame(pairs_rows, columns=["incoming_record_id","db_record_id","is_same_person"])

print("Pairs rows:", pairs_index.shape)
print("Missed positives (true match not in candidate set):", missed_pos)
print("Positive rate:", pairs_index["is_same_person"].mean())

# ---- Materialize left_/right_ columns (pairs-style) ----
inc_cols = ["incoming_record_id","patient_name","patient_dob","patient_doi","patient_ssn","claim_id","indexor_code"]
db_cols  = ["db_record_id","patient_name","patient_dob","patient_doi","patient_ssn","claim_id","indexor_code"]

pairs_full = pairs_index.merge(inc[inc_cols], on="incoming_record_id", how="left")
pairs_full = pairs_full.merge(db[db_cols], on="db_record_id", how="left", suffixes=("_left","_right"))

pairs_full = pairs_full.rename(columns={
    "patient_name_left":"left_name",
    "patient_dob_left":"left_dob",
    "patient_doi_left":"left_doi",
    "patient_ssn_left":"left_ssn",
    "claim_id_left":"left_claim",
    "indexor_code_left":"left_indexor_code",
    "patient_name_right":"right_name",
    "patient_dob_right":"right_dob",
    "patient_doi_right":"right_doi",
    "patient_ssn_right":"right_ssn",
    "claim_id_right":"right_claim",
    "indexor_code_right":"right_indexor_code",
})

pairs_full = pairs_full[[
    "incoming_record_id","db_record_id",
    "left_indexor_code","right_indexor_code",
    "left_name","left_doi","left_dob","left_ssn","left_claim",
    "right_name","right_doi","right_dob","right_ssn","right_claim",
    "is_same_person"
]]

# ---- Train/test split by incoming_record_id (no leakage) ----
incoming_ids = pairs_full["incoming_record_id"].unique()
rng.shuffle(incoming_ids)

cut = int(len(incoming_ids) * TRAIN_FRAC)
train_ids = set(incoming_ids[:cut])
test_ids  = set(incoming_ids[cut:])

pairs_train = pairs_full[pairs_full["incoming_record_id"].isin(train_ids)].reset_index(drop=True)
pairs_test  = pairs_full[pairs_full["incoming_record_id"].isin(test_ids)].reset_index(drop=True)

print("pairs_train:", pairs_train.shape, "pos_rate:", pairs_train["is_same_person"].mean())
print("pairs_test :", pairs_test.shape,  "pos_rate:", pairs_test["is_same_person"].mean())

# ---- Save to compressed CSV ----
OUT_TRAIN = "pairs_train_from_candidates.csv.gz"
OUT_TEST  = "pairs_test_from_candidates.csv.gz"

pairs_train.to_csv(OUT_TRAIN, index=False, compression="gzip")
pairs_test.to_csv(OUT_TEST, index=False, compression="gzip")

print("Wrote:", OUT_TRAIN, OUT_TEST)


Pairs rows: (2531677, 3)
Missed positives (true match not in candidate set): 246
Positive rate: 0.019652586013144648
pairs_train: (2025341, 15) pos_rate: 0.01965249308634941
pairs_test : (506336, 15) pos_rate: 0.019652957719775013
Wrote: pairs_train_from_candidates.csv.gz pairs_test_from_candidates.csv.gz


# Proceeding with feature engineering

### Functions used to normalize and compute similarity scores and other features for all columns

In [28]:
# convert all dates into one format
def parse_date(x):
    if pd.isna(x) or x in ("", "N/A", "NA", None):
        return None
    s = str(x)
    try:
        return parser.parse(s, fuzzy=True, dayfirst=False).date()
    except Exception:
        return None

# convert all names into one format (lowercased and alphanumeric characters + spaces only)
def normalize_name(s):
    if pd.isna(s) or not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = s.replace("'", " ").replace("-", " ")
    s = "".join(ch if ch.isalnum() or ch.isspace() else " " for ch in s)
    return " ".join(s.split())

def name_similarity(a, b):
    a, b = normalize_name(a), normalize_name(b)
    if not a and not b:
        return 0.5
    s1 = fuzz.token_set_ratio(a, b) / 100.0 # Order insensitivity score
    return s1

def date_similarity(d1, d2):
    d1, d2 = parse_date(d1), parse_date(d2)
    if d1 is None or d2 is None:
        return np.nan
    delta_days = abs((d1 - d2).days)
    return delta_days

def date_threshold_features(d1, d2):
    d1, d2 = parse_date(d1), parse_date(d2)
    if d1 is None or d2 is None:
        return np.nan, np.nan, np.nan

    delta = abs((d1 - d2).days)

    within_7 = int(delta <= 7)
    return within_7

def exact_month_match(d1, d2):
    d1, d2 = parse_date(d1), parse_date(d2)
    if d1 is None or d2 is None:
        return np.nan
    return int(d1.month == d2.month)

def exact_day_match(d1, d2):
    d1, d2 = parse_date(d1), parse_date(d2)
    if d1 is None or d2 is None:
        return np.nan
    return int(d1.day == d2.day)

# used for normalizing SSNs and claim #s by only returning lowercased alphanumeric characters and nothing else
def normalize_digits(x):
    if pd.isna(x):
        return ""
    s = str(x)
    return "".join(ch for ch in s if ch.isalnum()).lower()

def hamming_dist(a, b): # use for ssn
    a, b = normalize_digits(a), normalize_digits(b)
    if len(a) != len(b): return np.nan
    return sum(x != y for x, y in zip(a, b))

def exact_match(a, b): # use for both claim and ssn
    a, b = normalize_digits(a), normalize_digits(b)
    return int(a == b and a != "")

def levenshtein_dist(a, b): # use for both claim and ssn
    a, b = normalize_digits(a), normalize_digits(b)
    if not a or not b:
        return np.nan
    return Levenshtein.distance(a, b)

def ssn_position_features(a, b): # use for ssn
    a, b = normalize_digits(a), normalize_digits(b)
    if len(a) != 9 or len(b) != 9:
        return np.nan
        
    last4 = int(a[5:] == b[5:])

    return last4

### Create all feature columns here

In [30]:
def compute_feature_row(row):
    features = {}

    # -------------------
    # NAME FEATURES
    # -------------------
    n1 = name_similarity(row["left_name"], row["right_name"])
    features["name_token_set_sim"] = n1

    # -------------------
    # DATE FEATURES (DOI, DOB)
    # -------------------
    for prefix in ["doi", "dob"]:
        left_date = row[f"left_{prefix}"]
        right_date = row[f"right_{prefix}"]

        # continuous delta
        features[f"{prefix}_delta_days"] = date_similarity(left_date, right_date)

        # threshold flags
        w7 = date_threshold_features(left_date, right_date)
        features[f"{prefix}_within_7"] = w7

        # component matches
        features[f"{prefix}_month_match"] = exact_month_match(left_date, right_date)
        features[f"{prefix}_day_match"] = exact_day_match(left_date, right_date)

    # -------------------
    # SSN FEATURES
    # -------------------
    last4 = ssn_position_features(row["left_ssn"], row["right_ssn"])
    features["ssn_last4_match"] = last4

    # -------------------
    # CLAIM FEATURES
    # -------------------
    features["claim_exact_match"] = exact_match(row["left_claim"], row["right_claim"])
    features["claim_levenshtein"] = levenshtein_dist(row["left_claim"], row["right_claim"])
    features["claim_hamming_dist"] = hamming_dist(row["left_claim"], row["right_claim"])

    # Indexor codes exact match feature
    features["indexor_code_exact_match"] = exact_match(row["left_indexor_code"], row["right_indexor_code"])

    return pd.Series(features)

# Transform into dataframe

# COMPUTE FUNCTIONS SO THAT DOB AND SSNs ARE AN EXACT MATCH

# ALSO COMPUTE WHETHER DOB AND SSN IS ONE DIGIT OFF

# USE JARO WINKLER FOR SSNs AND NAMES

In [ ]:

# ============================================================
# Feature Engineering (RAM-safe)
# ============================================================
# This computes features in CHUNKS and writes them to disk incrementally.
# It also pre-parses dates once per chunk to avoid calling parse_date() many times per row.

import pandas as pd
import numpy as np
import re
from dateutil import parser

# ---- Tunables ----
CHUNK_SIZE = 10_000          # lower if you still hit RAM (e.g., 20_000)
OUT_TRAIN_FEATS = "train_features.csv.gz"
OUT_TEST_FEATS  = "test_features.csv.gz"

# ---- Safe parsers / normalizers ----
def safe_parse_date(x):
    if pd.isna(x) or x in ("", "N/A", "NA", None):
        return pd.NaT
    try:
        return parser.parse(str(x), fuzzy=True, dayfirst=False).date()
    except Exception:
        return pd.NaT

def normalize_name_series(s):
    s = s.fillna("").astype(str).str.strip().str.lower()
    s = s.str.replace("'", " ", regex=False).str.replace("-", " ", regex=False)
    s = s.str.replace(r"[^a-z0-9 ]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def normalize_digits_series(s):
    s = s.fillna("").astype(str).str.lower()
    s = s.str.replace(r"[^a-z0-9]", "", regex=True)
    return s

def ssn_position_features_vec(a, b): # compute threshold of match
    # expects normalized digit strings
    ok = (a.str.len() == 9) & (b.str.len() == 9)
    last4 = np.where(ok, (a.str.slice(5,9) == b.str.slice(5,9)).astype(int), np.nan)
    return last4

def hamming_dist_vec(a, b):
    # expects normalized strings; if lengths differ -> NaN
    la = a.str.len()
    lb = b.str.len()
    same_len = la == lb
    out = np.full(len(a), np.nan, dtype=float)
    idx = np.where(same_len.values)[0]
    for i in idx:
        out[i] = sum(ch1 != ch2 for ch1, ch2 in zip(a.iat[i], b.iat[i]))
    return out

def date_features_vec(left_series, right_series, prefix):
    # robust parse once
    ldt = pd.to_datetime(left_series.map(safe_parse_date), errors="coerce")
    rdt = pd.to_datetime(right_series.map(safe_parse_date), errors="coerce")

    delta = (ldt - rdt).abs().dt.days
    out = pd.DataFrame({
        f"{prefix}_delta_days": delta,
        f"{prefix}_within_7": (delta <= 7).astype("float").where(delta.notna()),
        f"{prefix}_month_match": (ldt.dt.month == rdt.dt.month).astype("float").where(delta.notna()),
        f"{prefix}_day_match": (ldt.dt.day == rdt.dt.day).astype("float").where(delta.notna()),
    })
    return out

# ---- Name similarity: prefer rapidfuzz if available, else fallback to fuzzywuzzy ----
try:
    from rapidfuzz import fuzz as rfuzz
    _use_rapidfuzz = True
except Exception:
    _use_rapidfuzz = False
    from fuzzywuzzy import fuzz as fwuzz

def name_sims_vec(a, b):
    a = normalize_name_series(a)
    b = normalize_name_series(b)
    # handle both empty -> 0.5 (your convention)
    both_empty = (a == "") & (b == "")
    if _use_rapidfuzz:
        token_set = np.array([rfuzz.token_set_ratio(x,y)/100.0 for x,y in zip(a,b)], dtype=float)
    else:
        token_set = np.array([fwuzz.token_set_ratio(x,y)/100.0 for x,y in zip(a,b)], dtype=float)
    token_set[both_empty.values] = 0.5
    return token_set

# ---- Levenshtein: try rapidfuzz.distance first (fast), else keep your original libs if present ----
try:
    from rapidfuzz.distance import Levenshtein as RLev
    _use_rf_dist = True
except Exception:
    _use_rf_dist = False


def levenshtein_vec(a, b):
    a = normalize_digits_series(a)
    b = normalize_digits_series(b)
    out = np.full(len(a), np.nan, dtype=float)
    ok = (a != "") & (b != "")
    idx = np.where(ok.values)[0]
    if _use_rf_dist:
        for i in idx:
            out[i] = RLev.distance(a.iat[i], b.iat[i])
    else:
        for i in idx:
            out[i] = Levenshtein.distance(a.iat[i], b.iat[i])
    return out

def exact_match_vec(a, b):
    a = normalize_digits_series(a)
    b = normalize_digits_series(b)
    return ((a == b) & (a != "")).astype(int)

def build_features_chunk(df_chunk: pd.DataFrame) -> pd.DataFrame:
    feats = pd.DataFrame(index=df_chunk.index)

    # Name sims
    ts = name_sims_vec(df_chunk["left_name"], df_chunk["right_name"])
    feats["name_token_set_sim"] = ts

    # Date features
    for pref in ["doi", "dob"]:
        feats = pd.concat([feats, date_features_vec(df_chunk[f"left_{pref}"], df_chunk[f"right_{pref}"], pref)], axis=1)

    # SSN features
    lssn = df_chunk["left_ssn"]
    rssn = df_chunk["right_ssn"]
    l4 = ssn_position_features_vec(normalize_digits_series(lssn), normalize_digits_series(rssn))
    feats["ssn_last4_match"] = l4

    # Claim features
    lclm = df_chunk["left_claim"]
    rclm = df_chunk["right_claim"]
    feats["claim_exact_match"] = exact_match_vec(lclm, rclm)
    feats["claim_levenshtein"] = levenshtein_vec(lclm, rclm)
    feats["claim_hamming_dist"] = hamming_dist_vec(normalize_digits_series(lclm), normalize_digits_series(rclm))

    # Indexor exact (should always be 1 after blocking, but keep it as a guardrail feature)
    feats["indexor_code_exact_match"] = exact_match_vec(df_chunk["left_indexor_code"], df_chunk["right_indexor_code"])

    return feats

def featurize_in_chunks(pairs_df: pd.DataFrame, out_path: str):
    first = True
    for start in range(0, len(pairs_df), CHUNK_SIZE):
        chunk = pairs_df.iloc[start:start+CHUNK_SIZE].copy()
        fdf = build_features_chunk(chunk)
        fdf["is_same_person"] = chunk["is_same_person"].values
        # append to disk
        fdf.to_csv(out_path, index=False, mode="wt" if first else "at",
                   header=first, compression="gzip")
        first = False
        print(f"Wrote features rows {start:,}..{min(start+CHUNK_SIZE, len(pairs_df)):,} -> {out_path}")

# Run featurization
featurize_in_chunks(pairs_train, OUT_TRAIN_FEATS)
featurize_in_chunks(pairs_test, OUT_TEST_FEATS)

print("Done. Outputs:")
print(" -", OUT_TRAIN_FEATS)
print(" -", OUT_TEST_FEATS)

Wrote features rows 0..10,000 -> train_features.csv.gz
Wrote features rows 10,000..20,000 -> train_features.csv.gz
Wrote features rows 20,000..30,000 -> train_features.csv.gz
Wrote features rows 30,000..40,000 -> train_features.csv.gz
Wrote features rows 40,000..50,000 -> train_features.csv.gz
Wrote features rows 50,000..60,000 -> train_features.csv.gz
Wrote features rows 60,000..70,000 -> train_features.csv.gz
Wrote features rows 70,000..80,000 -> train_features.csv.gz
Wrote features rows 80,000..90,000 -> train_features.csv.gz
Wrote features rows 90,000..100,000 -> train_features.csv.gz
Wrote features rows 100,000..110,000 -> train_features.csv.gz
Wrote features rows 110,000..120,000 -> train_features.csv.gz
Wrote features rows 120,000..130,000 -> train_features.csv.gz
Wrote features rows 130,000..140,000 -> train_features.csv.gz
Wrote features rows 140,000..150,000 -> train_features.csv.gz
Wrote features rows 150,000..160,000 -> train_features.csv.gz
Wrote features rows 160,000..170

# Set up ML model

### Read data

In [35]:
import pandas as pd

train_feats = pd.read_csv("train_features.csv.gz")
test_feats  = pd.read_csv("test_features.csv.gz")

### Setting up train and test data

In [37]:
X_train = train_feats.drop(columns=["is_same_person"])
y_train = train_feats["is_same_person"]

X_test = test_feats.drop(columns=["is_same_person"])
y_test = test_feats["is_same_person"]

In [38]:
feature_cols = X_train.columns.tolist()

In [116]:
X_train.columns

Index(['name_token_set_sim', 'doi_delta_days', 'doi_within_7',
       'doi_month_match', 'doi_day_match', 'dob_delta_days', 'dob_within_7',
       'dob_month_match', 'dob_day_match', 'ssn_last4_match',
       'claim_exact_match', 'claim_levenshtein', 'claim_hamming_dist',
       'indexor_code_exact_match'],
      dtype='object')

### Feature Correlation Analysis

Pairwise correlations between numeric features are examined to understand how strongly features are linearly related to one another.

Highly correlated features may indicate redundant information, which can lead to instability in some models and make interpretation more difficult. Correlation analysis helps identify features that may be capturing similar signals.

Because the model used here is regularized logistic regression, moderate correlations are generally not problematic; however, extremely high correlations may still indicate redundant feature engineering.

This analysis is therefore used as a diagnostic step to understand the structure of the feature space and identify potential redundancy among features.

In [40]:
# Correlation matrix (train only)
corr = X_train.corr(numeric_only=True)

# Show the most correlated pairs (excluding self)
corr_pairs = (
    corr.abs()
        .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .sort_values(ascending=False)
)

print("Top 25 absolute correlations:")
print(corr_pairs.head(25))

Top 25 absolute correlations:
claim_levenshtein   claim_hamming_dist    0.967287
dob_within_7        claim_exact_match     0.777551
                    claim_hamming_dist    0.770542
claim_exact_match   claim_hamming_dist    0.747882
ssn_last4_match     claim_exact_match     0.662650
dob_within_7        ssn_last4_match       0.655528
                    dob_day_match         0.654963
ssn_last4_match     claim_hamming_dist    0.595162
dob_within_7        claim_levenshtein     0.555249
claim_exact_match   claim_levenshtein     0.551798
doi_within_7        doi_month_match       0.549165
dob_day_match       claim_hamming_dist    0.534467
                    claim_exact_match     0.518211
dob_within_7        dob_month_match       0.481652
ssn_last4_match     claim_levenshtein     0.458937
name_token_set_sim  claim_exact_match     0.452614
dob_day_match       ssn_last4_match       0.431704
name_token_set_sim  doi_month_match       0.426337
                    doi_delta_days        0.403272
 

### Variance Inflation Factor (VIF)

Variance Inflation Factor (VIF) is used to quantify multicollinearity among features. It measures how much the variance of a feature's estimated coefficient increases due to linear relationships with other features.

Higher VIF values indicate stronger multicollinearity:

- VIF ≈ 1 → little to no correlation with other features
- VIF bet6een 1–5 → moderate correlation but typically acceptable
- VIF > 10 → strong multicollinearity that may warrant investigation

The model used in this analysis is logistic regression with **L1 regularization**, which encourages sparsity in the model coefficients and can effectively perform feature selection. When multiple features capture similar information, L1 regularization tends to retain the most predictive feature while shrinking redundant features toward zero.

The VIF analysis is therefore used as a diagnostic tool to understand the degree of redundancy among engineered features and to identify cases where multiple features may be capturing the same underlying signal.estimates.

In [42]:
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def compute_vif(df):
    # Impute + scale first (important)
    X = SimpleImputer(strategy="median").fit_transform(df)
    X = StandardScaler().fit_transform(X)

    vifs = []
    for i in range(X.shape[1]):
        y = X[:, i]
        X_others = np.delete(X, i, axis=1)

        lr = LinearRegression()
        lr.fit(X_others, y)
        r2 = lr.score(X_others, y)

        # Avoid divide-by-zero
        vif = np.inf if (1 - r2) < 1e-12 else 1.0 / (1.0 - r2)
        vifs.append(vif)

    return pd.DataFrame({"feature": df.columns, "VIF": vifs}).sort_values("VIF", ascending=False)

vif_df = compute_vif(X_train)
vif_df.head(20)

,feature,VIF
13,indexor_code_exact_match,inf
6,dob_within_7,3.954722
10,claim_exact_match,3.670429
12,claim_hamming_dist,3.580589
11,claim_levenshtein,2.228334
0,name_token_set_sim,2.151834
3,doi_month_match,1.847203
8,dob_day_match,1.752513
9,ssn_last4_match,1.658320
2,doi_within_7,1.552084


### Interpretation

The VIF results suggest that most engineered features exhibit low to moderate multicollinearity, with the majority falling below 5. This indicates that, although some features capture related aspects of similarity, multicollinearity is not broadly severe across the feature set.

`claim_exact_match` shows the highest finite VIF (5.51), which is expected given the presence of other claim-based similarity features such as `claim_hamming_dist` and `claim_levenshtein`. This suggests some redundancy within the claim-related feature group, but not to a degree that is unexpected for record linkage features.

`dob_within_7` also shows moderate multicollinearity (VIF 4.49), which is consistent with the inclusion of related date-of-birth features such as `dob_day_match`, `dob_month_match`, and `dob_delta_days`. These features are intentionally related and represent different degrees of agreement on the same underlying field.

`indexor_code_exact_match` has an infinite VIF, indicating perfect or near-perfect linear dependence with one or more other features, or a near-constant/separable feature pattern in the sample. This suggests that the feature may be redundant or non-informative in the current dataset.

Overall, the VIF analysis suggests that feature redundancy exists in a few expected areas, but widespread multicollinearity is not a major concern. Because the final model uses L1-regularized logistic regression, some redundancy is acceptable, as the regularization can shrink less useful correlated features toward zero.

### Chi-Squared Feature Association Test

The chi-squared test is used to evaluate the statistical association between binary features and the target label (true match vs. non-match).

The test compares the observed distribution of feature values across the match and non-match classes with the distribution that would be expected if the feature were independent of the target.

For each feature, the test produces:

- **Chi-squared statistic (χ²)** – measures the magnitude of the association between the feature and the target label.
- **p-value** – indicates the probability of observing a relationship this strong if the feature and label were actually independent.

Large chi-squared values combined with very small p-values indicate strong evidence that the feature contains information useful for distinguishing matches from non-matches.

This test is used as a statistical sanity check to confirm that engineered matching features contain meaningful signal relative to the true match label.

In [45]:
from sklearn.feature_selection import chi2

# Identify mostly-binary columns (values subset of {0,1} ignoring NaNs)
binary_cols = []
for c in X_train.columns:
    vals = pd.Series(X_train[c].dropna().unique())
    if len(vals) > 0 and vals.isin([0, 1]).all():
        binary_cols.append(c)

print("Binary cols:", binary_cols)

Xb_train = X_train[binary_cols].fillna(0)  # chi2 needs non-negative
chi2_stats, pvals = chi2(Xb_train, y_train)

chi_df = pd.DataFrame({
    "feature": binary_cols,
    "chi2": chi2_stats,
    "p_value": pvals
}).sort_values("chi2", ascending=False)

chi_df.head(20)

Binary cols: ['doi_within_7', 'doi_month_match', 'doi_day_match', 'dob_within_7', 'dob_month_match', 'dob_day_match', 'ssn_last4_match', 'claim_exact_match', 'indexor_code_exact_match']


,feature,chi2,p_value
7,claim_exact_match,1.696524e+06,0.0
3,dob_within_7,1.384527e+06,0.0
6,ssn_last4_match,6.220938e+05,0.0
5,dob_day_match,5.983403e+05,0.0
4,dob_month_match,2.965013e+05,0.0
0,doi_within_7,1.311405e+05,0.0
1,doi_month_match,1.912152e+04,0.0
2,doi_day_match,2.109126e+03,0.0
8,indexor_code_exact_match,0.000000e+00,1.0


### Interpretation

The chi-squared results show extremely strong statistical association between several engineered matching features and the true match label. In particular, claim identifier agreement, date-of-birth proximity, and SSN last-four agreement exhibit the strongest relationships.

These results confirm that the engineered features contain meaningful signal that can be leveraged by the machine learning model to distinguish true matches from candidate non-matches.

### Logistic Regression

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is a good choice for small datasets
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1] # KEEP AT THE ROW LEVEL AND ANALYZE

# Convert probabilities to binary predictions using a threshold (e.g., 0.5)
y_pred = (y_pred_proba > 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"Average Precision Score: {avg_precision:.4f}")
print("Classification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

ROC AUC Score: 0.9997
Average Precision Score: 0.9889
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    496385
           1       0.91      0.96      0.93      9951

    accuracy                           1.00    506336
   macro avg       0.95      0.98      0.97    506336
weighted avg       1.00      1.00      1.00    506336

Confusion matrix:
[[495438    947]
 [   410   9541]]


### Logistic regression hyperparameter tuning

# USE GBT
# INSTEAD OF GRID USE RANDOM SEARCH

In [50]:
# from sklearn.model_selection import GridSearchCV

# param_grid = {
#     'C': [0.001, 0.01, 0.1, 1, 10, 100],
#     'penalty': ['l1', 'l2'],
#     'solver': ['liblinear']
# }

# print("Parameter grid defined successfully:")
# print(param_grid)

In [51]:
# grid_search = GridSearchCV(LogisticRegression(random_state=42, solver='liblinear'), param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
# grid_search.fit(X_train, y_train)

# print("GridSearchCV fitting complete.")

In [52]:
# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best score (ROC AUC): {grid_search.best_score_:.4f}")

In [53]:
# best_model = LogisticRegression(random_state=42, **grid_search.best_params_)
# best_model.fit(X_train, y_train)

# y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]
# y_pred_tuned = (y_pred_proba_tuned > 0.5).astype(int)

# roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)
# avg_precision_tuned = average_precision_score(y_test, y_pred_proba_tuned)

# print(f"Tuned Model ROC AUC Score: {roc_auc_tuned:.4f}")
# print(f"Tuned Model Average Precision Score: {avg_precision_tuned:.4f}")
# print("Tuned Model Classification report:")
# print(classification_report(y_test, y_pred_tuned))
# print("Tuned Model Confusion matrix:")
# print(confusion_matrix(y_test, y_pred_tuned))

# print("\n--- Performance Comparison ---")
# print(f"Untuned Model ROC AUC: {roc_auc:.4f}")
# print(f"Tuned Model ROC AUC: {roc_auc_tuned:.4f}")
# print(f"Improvement in ROC AUC: {roc_auc_tuned - roc_auc:.4f}")

# print(f"Untuned Model Average Precision: {avg_precision:.4f}")
# print(f"Tuned Model Average Precision: {avg_precision_tuned:.4f}")
# print(f"Improvement in Average Precision: {avg_precision_tuned - avg_precision:.4f}")

### Focus on minimizing false positives (precision vs loss?)

In [55]:
# grid_search = GridSearchCV(LogisticRegression(random_state=42, solver='liblinear'), param_grid, cv=5, scoring='precision', n_jobs=-1) #losses
# grid_search.fit(X_train, y_train)

# print("GridSearchCV fitting complete.")

In [56]:
# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best score (ROC AUC): {grid_search.best_score_:.4f}")

In [57]:
# best_model = LogisticRegression(random_state=42, **grid_search.best_params_)
# best_model.fit(X_train, y_train)

# y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]
# y_pred_tuned = (y_pred_proba_tuned > 0.5).astype(int)

# roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)
# avg_precision_tuned = average_precision_score(y_test, y_pred_proba_tuned)

# print(f"Tuned Model ROC AUC Score: {roc_auc_tuned:.4f}")
# print(f"Tuned Model Average Precision Score: {avg_precision_tuned:.4f}")
# print("Tuned Model Classification report:")
# print(classification_report(y_test, y_pred_tuned))
# print("Tuned Model Confusion matrix:")
# print(confusion_matrix(y_test, y_pred_tuned))

# print("\n--- Performance Comparison ---")
# print(f"Untuned Model ROC AUC: {roc_auc:.4f}")
# print(f"Tuned Model ROC AUC: {roc_auc_tuned:.4f}")
# print(f"Improvement in ROC AUC: {roc_auc_tuned - roc_auc:.4f}")

# print(f"Untuned Model Average Precision: {avg_precision:.4f}")
# print(f"Tuned Model Average Precision: {avg_precision_tuned:.4f}")
# print(f"Improvement in Average Precision: {avg_precision_tuned - avg_precision:.4f}")

### Best model (to not have to run grid search again)

In [59]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Fill missing values
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Model with best parameters
model = LogisticRegression(
    C=0.01,
    penalty='l1',
    solver='liblinear',
    random_state=42,
)

model.fit(X_train, y_train)

# Predicted probabilities
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Binary predictions (default threshold = 0.5)
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"Average Precision Score: {avg_precision:.4f}")
print("Classification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

ROC AUC Score: 0.9999
Average Precision Score: 0.9963
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    496385
           1       0.95      0.99      0.97      9951

    accuracy                           1.00    506336
   macro avg       0.98      1.00      0.99    506336
weighted avg       1.00      1.00      1.00    506336

Confusion matrix:
[[495874    511]
 [    81   9870]]


### Inspecting coefficients of logistic regression

In [61]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# X_train, y_train assumed:
# X_train = training_feature_df.drop(columns=["is_same_person"])
# y_train = training_feature_df["is_same_person"]

feature_cols = X_train.columns.tolist()

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        C=0.01,
        penalty="l1",
        solver="liblinear",
        max_iter=2000
    ))
])

pipe.fit(X_train, y_train)

coef = pipe.named_steps["lr"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": coef,
    "abs_coef": np.abs(coef)
}).sort_values("abs_coef", ascending=False)

coef_df.head(20)

,feature,coef,abs_coef
2,doi_within_7,1.990580,1.990580
10,claim_exact_match,1.282151,1.282151
6,dob_within_7,1.066676,1.066676
8,dob_day_match,0.488050,0.488050
0,name_token_set_sim,0.401884,0.401884
9,ssn_last4_match,-0.297678,0.297678
11,claim_levenshtein,0.266560,0.266560
12,claim_hamming_dist,-0.244471,0.244471
4,doi_day_match,0.010940,0.010940
1,doi_delta_days,0.000000,0.000000


### Interpretation of Model Coefficients

The fitted L1-regularized logistic regression coefficients indicate which engineered features were retained by the model and the direction of their association with the match label.

Positive coefficients indicate that higher values of the feature increase the model’s estimated probability that a candidate pair is a true match, while negative coefficients indicate that higher values decrease that probability. Features with coefficients shrunk to exactly zero were removed by the L1 penalty, indicating that they did not provide additional predictive information beyond the features already retained.

The strongest positive coefficients were observed for `doi_within_7`, `claim_exact_match`, `dob_within_7`, and `name_token_set_sim`. These features therefore represent the primary signals used by the model to identify likely matches.

Several related date-based features (`doi_day_match`, `doi_month_match`, `dob_month_match`, `dob_delta_days`) were shrunk to zero. This is expected when using L1 regularization in the presence of correlated predictors: the model tends to retain the most predictive representative of a group of related features while eliminating redundant alternatives.

The coefficient for `ssn_last4_match` appears negative in the fitted model. While this may seem counterintuitive, coefficients in a regularized model should be interpreted conditionally rather than in isolation. Because several stronger identity signals (such as claim identifier agreement, DOB proximity, and name similarity) are already captured by other retained features, the coefficient reflects the incremental effect of SSN agreement *after accounting for those signals*. In correlated feature sets, L1 regularization can assign signs that differ from the marginal relationship of the feature with the label.

Importantly, earlier feature diagnostics (such as the chi-squared test) show that SSN last-four agreement still has strong marginal association with true matches, indicating that the feature contains meaningful signal even if its final coefficient sign is influenced by interactions with other predictors.

Overall, the coefficient pattern indicates that the model relies on a compact set of strong matching signals while eliminating redundant features through L1 regularization.

### Ablation (getting rid of hard columns)

In [64]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

def train_eval(feature_subset, name):
    Xtr = X_train[feature_subset]
    Xte = X_test[feature_subset]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, n_jobs=-1))
    ])
    pipe.fit(Xtr, y_train)

    proba = pipe.predict_proba(Xte)[:,1]
    pred = (proba >= 0.5).astype(int)

    print(f"\n===== {name} =====")
    print("ROC AUC:", roc_auc_score(y_test, proba))
    print("Avg Precision:", average_precision_score(y_test, proba))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=4))

    # coefficient peek (top 15)
    coef = pipe.named_steps["lr"].coef_.ravel()
    coef_df = pd.DataFrame({"feature": feature_subset, "coef": coef, "abs": np.abs(coef)}).sort_values("abs", ascending=False)
    print("\nTop coefficients:")
    print(coef_df.head(15).to_string(index=False))

# --- Define feature subsets ---
all_feats = X_train.columns.tolist()

no_hard_ids = [f for f in all_feats if f not in {
    "claim_exact_match"
}]

soft_only = [f for f in all_feats if f not in {
    "claim_exact_match",
    "ssn_last4_match",
    "doi_within_7",
    "dob_within_7",
    "doi_month_match","doi_day_match",
    "dob_month_match","dob_day_match",
    "indexor_code_exact_match"
}]

# Run them
train_eval(all_feats, "A) Full feature set")
train_eval(no_hard_ids, "B) Drop claim_exact_match")
train_eval(soft_only, "C) Mostly soft signals only")


===== A) Full feature set =====
ROC AUC: 0.9999291007032801
Avg Precision: 0.9964716667040118
Confusion matrix:
 [[495912    473]
 [    70   9881]]
              precision    recall  f1-score   support

           0     0.9999    0.9990    0.9995    496385
           1     0.9543    0.9930    0.9733      9951

    accuracy                         0.9989    506336
   macro avg     0.9771    0.9960    0.9864    506336
weighted avg     0.9990    0.9989    0.9989    506336


Top coefficients:
                 feature      coef      abs
          dob_delta_days -3.421665 3.421665
            doi_within_7  2.825206 2.825206
       claim_exact_match  1.548199 1.548199
            dob_within_7  0.962083 0.962083
          doi_delta_days -0.847699 0.847699
           dob_day_match  0.797807 0.797807
      claim_hamming_dist -0.464642 0.464642
       claim_levenshtein  0.451874 0.451874
      name_token_set_sim  0.415051 0.415051
         ssn_last4_match -0.310748 0.310748
         dob_month_ma

## End-to-end comparison (Business vs ML reranker) + hard-slice recovery

This section evaluates the trained ML model as a **reranker** on the **full candidate set** (all `candidates_pruned` rows per `incoming_record_id`), and compares:

- **Business baseline:** match rate from the lookup logic (1A/1B/2)
- **ML reranker:** Top-1 accuracy (choose the highest-scored candidate per incoming)

It also reports ML recovery specifically on the **subset the business logic fails**.

In [66]:
# ============================================================
# Business vs ML reranker (full candidate set)
# ============================================================
import numpy as np
import pandas as pd

assert "db" in globals(), "Expected dataframe `db`."
assert "inc" in globals(), "Expected dataframe `incoming`."
assert "truth" in globals(), "Expected dataframe `ground_truth`."
assert "candidates_pruned" in globals(), "Expected `candidates_pruned` from blocking."

# Identify the fitted model object
_model = None
if "pipe" in globals():
    _model = pipe
elif "best_model" in globals():
    _model = best_model
else:
    raise AssertionError("Expected a fitted model `pipe` (Pipeline) or `best_model` (LogisticRegression).")

# Build full candidate pairs
pairs_all = candidates_pruned.merge(
    truth[["incoming_record_id","db_record_id"]].rename(columns={"db_record_id":"true_db_record_id"}),
    on="incoming_record_id",
    how="left"
)
pairs_all["is_same_person"] = (pairs_all["db_record_id"] == pairs_all["true_db_record_id"]).astype(int)

pairs_all = pairs_all.merge(
    inc.rename(columns={
        "patient_name":"left_name","patient_dob":"left_dob","patient_doi":"left_doi",
        "patient_ssn":"left_ssn","claim_id":"left_claim","indexor_code":"left_indexor_code"
    })[["incoming_record_id","left_name","left_dob","left_doi","left_ssn","left_claim","left_indexor_code"]],
    on="incoming_record_id",
    how="left"
).merge(
    db.rename(columns={
        "patient_name":"right_name","patient_dob":"right_dob","patient_doi":"right_doi",
        "patient_ssn":"right_ssn","claim_id":"right_claim","indexor_code":"right_indexor_code"
    })[["db_record_id","right_name","right_dob","right_doi","right_ssn","right_claim","right_indexor_code"]],
    on="db_record_id",
    how="left"
)

print("Avg candidates per incoming:", pairs_all.groupby("incoming_record_id").size().mean())
print("Candidate recall:", pairs_all.groupby("incoming_record_id")["is_same_person"].max().mean())

# ---- Chunked scoring ----
CHUNK = 20_000

def _predict_proba(X):
    return _model.predict_proba(X)[:, 1]

_expected_cols = None
if "X_train" in globals():
    _expected_cols = list(X_train.columns)
elif "training_feature_df" in globals():
    _expected_cols = [c for c in training_feature_df.columns if c != "is_same_person"]

scored_parts = []
for start in range(0, len(pairs_all), CHUNK):
    chunk = pairs_all.iloc[start:start+CHUNK].copy()

    if "build_features_chunk" in globals():
        feats = build_features_chunk(chunk)
    elif "compute_feature_row" in globals():
        feats = chunk.apply(compute_feature_row, axis=1)
    else:
        raise AssertionError("Need either `build_features_chunk` or `compute_feature_row` in scope.")

    if _expected_cols is None:
        _expected_cols = list(feats.columns)
    else:
        for c in _expected_cols:
            if c not in feats.columns:
                feats[c] = np.nan
        feats = feats[_expected_cols]

    proba = _predict_proba(feats)

    scored_parts.append(pd.DataFrame({
        "incoming_record_id": chunk["incoming_record_id"].values,
        "db_record_id": chunk["db_record_id"].values,
        "is_same_person": chunk["is_same_person"].values,
        "proba": proba
    }))
    print(f"Scored rows {start:,}..{min(start+CHUNK, len(pairs_all)):,}")

scored_all = pd.concat(scored_parts, ignore_index=True)

top1_all = (scored_all.sort_values(["incoming_record_id","proba"], ascending=[True, False])
                     .groupby("incoming_record_id", as_index=False)
                     .head(1))

ml_top1 = top1_all["is_same_person"].mean()
print("\nML Top-1 accuracy on FULL candidate set:", ml_top1)
print("Incoming evaluated:", top1_all["incoming_record_id"].nunique())

# ---- Business baseline ----
business_recall = None
if "true" in globals() and "correct_match" in true.columns:
    business_recall = float(true["correct_match"].mean())
elif "eval_df" in globals() and ("pred_db_record_id" in eval_df.columns) and ("db_record_id" in eval_df.columns):
    tmp_eval = eval_df.copy()
    tmp_eval["pred_db_int"] = pd.to_numeric(tmp_eval["pred_db_record_id"], errors="coerce")
    tmp_eval["truth_db_int"] = pd.to_numeric(tmp_eval["db_record_id"], errors="coerce")
    tmp_eval["correct_match"] = ((tmp_eval["pred_db_int"].notna()) & (tmp_eval["pred_db_int"] == tmp_eval["truth_db_int"])).astype(int)
    business_recall = float(tmp_eval["correct_match"].mean())

if business_recall is not None:
    print("\nBusiness match rate:", business_recall)
    print("Absolute lift:", ml_top1 - business_recall)
    print("Relative lift:", (ml_top1 / business_recall) - 1)
else:
    print("\n[Note] Business baseline not found (`true` or `eval_df`).")

# ---- Hard-slice recovery ----
fail_ids = None
if "true" in globals() and "correct_match" in true.columns:
    fail_ids = set(true.loc[true["correct_match"] == 0, "incoming_record_id"])
elif "eval_df" in globals() and business_recall is not None:
    fail_ids = set(tmp_eval.loc[tmp_eval["correct_match"] == 0, "incoming_record_id"])

if fail_ids is not None and len(fail_ids) > 0:
    top1_fail = top1_all[top1_all["incoming_record_id"].isin(fail_ids)]
    recovered = float(top1_fail["is_same_person"].mean())
    print("\nOn BUSINESS-FAIL slice:")
    print("  Incoming in slice:", len(fail_ids))
    print("  ML Top-1 accuracy:", recovered)
    print("  Recovered matches:", int(top1_fail["is_same_person"].sum()), "/", len(top1_fail))
else:
    print("\n[Note] Could not compute business-fail slice.")

# ---- Optional: bucket breakdown if you have `unmatched` with `bucket` ----
if "unmatched" in globals() and "bucket" in unmatched.columns:
    slice_eval = unmatched[["incoming_record_id","bucket"]].merge(
        top1_all[["incoming_record_id","is_same_person"]],
        on="incoming_record_id",
        how="left"
    )
    print("\nML recovery by business failure bucket:")
    display(slice_eval.groupby("bucket")["is_same_person"].agg(["mean","count"]).sort_values("count", ascending=False))

Avg candidates per incoming: 155.9073
Candidate recall: 0.99508
Scored rows 0..20,000
Scored rows 20,000..40,000
Scored rows 40,000..60,000
Scored rows 60,000..80,000
Scored rows 80,000..100,000
Scored rows 100,000..120,000
Scored rows 120,000..140,000
Scored rows 140,000..160,000
Scored rows 160,000..180,000
Scored rows 180,000..200,000
Scored rows 200,000..220,000
Scored rows 220,000..240,000
Scored rows 240,000..260,000
Scored rows 260,000..280,000
Scored rows 280,000..300,000
Scored rows 300,000..320,000
Scored rows 320,000..340,000
Scored rows 340,000..360,000
Scored rows 360,000..380,000
Scored rows 380,000..400,000
Scored rows 400,000..420,000
Scored rows 420,000..440,000
Scored rows 440,000..460,000
Scored rows 460,000..480,000
Scored rows 480,000..500,000
Scored rows 500,000..520,000
Scored rows 520,000..540,000
Scored rows 540,000..560,000
Scored rows 560,000..580,000
Scored rows 580,000..600,000
Scored rows 600,000..620,000
Scored rows 620,000..640,000
Scored rows 640,000..6

In [111]:
# ============================================================
# Business vs ML reranker (TEST incoming records only)
# ============================================================
import numpy as np
import pandas as pd

assert "db" in globals(), "Expected dataframe `db`."
assert "inc" in globals(), "Expected dataframe `inc`."
assert "truth" in globals(), "Expected dataframe `truth`."
assert "candidates_pruned" in globals(), "Expected `candidates_pruned` from blocking."
assert "test_ids" in globals(), "Expected held-out `test_ids` from train/test split."

# ----------------------------
# 1) Pick fitted model
# ----------------------------
_model = None
if "pipe" in globals():
    _model = pipe
elif "best_model" in globals():
    _model = best_model
elif "model" in globals():
    _model = model
else:
    raise AssertionError("Expected a fitted model: `pipe`, `best_model`, or `model`.")

# ----------------------------
# 2) Restrict everything to TEST incoming IDs only
# ----------------------------
test_ids_list = list(test_ids)

truth_test = truth[truth["incoming_record_id"].isin(test_ids_list)].copy()
inc_test = inc[inc["incoming_record_id"].isin(test_ids_list)].copy()
candidates_test = candidates_pruned[candidates_pruned["incoming_record_id"].isin(test_ids_list)].copy()

# keep only true-match rows if truth has multiple row types
if "is_true_match" in truth_test.columns:
    truth_test = truth_test[truth_test["is_true_match"] == 1].copy()

# ----------------------------
# 3) Build test candidate pairs
# ----------------------------
pairs_test_all = candidates_test.merge(
    truth_test[["incoming_record_id", "db_record_id"]].rename(columns={"db_record_id": "true_db_record_id"}),
    on="incoming_record_id",
    how="left"
)

pairs_test_all["is_same_person"] = (
    pairs_test_all["db_record_id"] == pairs_test_all["true_db_record_id"]
).astype(int)

pairs_test_all = pairs_test_all.merge(
    inc_test.rename(columns={
        "patient_name": "left_name",
        "patient_dob": "left_dob",
        "patient_doi": "left_doi",
        "patient_ssn": "left_ssn",
        "claim_id": "left_claim",
        "indexor_code": "left_indexor_code"
    })[[
        "incoming_record_id",
        "left_name", "left_dob", "left_doi", "left_ssn",
        "left_claim", "left_indexor_code"
    ]],
    on="incoming_record_id",
    how="left"
).merge(
    db.rename(columns={
        "patient_name": "right_name",
        "patient_dob": "right_dob",
        "patient_doi": "right_doi",
        "patient_ssn": "right_ssn",
        "claim_id": "right_claim",
        "indexor_code": "right_indexor_code"
    })[[
        "db_record_id",
        "right_name", "right_dob", "right_doi", "right_ssn",
        "right_claim", "right_indexor_code"
    ]],
    on="db_record_id",
    how="left"
)

print("Held-out incoming records:", len(test_ids_list))
print("Avg candidates per incoming (TEST):",
      pairs_test_all.groupby("incoming_record_id").size().mean())
print("Candidate recall on TEST:",
      pairs_test_all.groupby("incoming_record_id")["is_same_person"].max().mean())

# ----------------------------
# 4) Score in chunks
# ----------------------------
CHUNK = 20_000

def _predict_proba(X):
    return _model.predict_proba(X)[:, 1]

_expected_cols = None
if "X_train" in globals():
    _expected_cols = list(X_train.columns)
elif "training_feature_df" in globals():
    _expected_cols = [c for c in training_feature_df.columns if c != "is_same_person"]

scored_parts = []

for start in range(0, len(pairs_test_all), CHUNK):
    chunk = pairs_test_all.iloc[start:start + CHUNK].copy()

    if "build_features_chunk" in globals():
        feats = build_features_chunk(chunk)
    elif "compute_feature_row" in globals():
        feats = chunk.apply(compute_feature_row, axis=1)
    else:
        raise AssertionError("Need either `build_features_chunk` or `compute_feature_row` in scope.")

    # align columns to training feature set
    if _expected_cols is None:
        _expected_cols = list(feats.columns)
    else:
        for c in _expected_cols:
            if c not in feats.columns:
                feats[c] = np.nan
        feats = feats[_expected_cols]

    proba = _predict_proba(feats)

    scored_parts.append(pd.DataFrame({
        "incoming_record_id": chunk["incoming_record_id"].values,
        "db_record_id": chunk["db_record_id"].values,
        "is_same_person": chunk["is_same_person"].values,
        "proba": proba
    }))

    print(f"Scored TEST rows {start:,}..{min(start + CHUNK, len(pairs_test_all)):,}")

scored_test = pd.concat(scored_parts, ignore_index=True)

# ----------------------------
# 5) Top-1 / Recall@K on TEST
# ----------------------------
scored_test = scored_test.sort_values(
    ["incoming_record_id", "proba"],
    ascending=[True, False]
).copy()

scored_test["rank"] = scored_test.groupby("incoming_record_id").cumcount() + 1

top1_test = scored_test[scored_test["rank"] == 1].copy()
ml_top1_test = top1_test["is_same_person"].mean()

recall_at_3_test = (
    scored_test[scored_test["rank"] <= 3]
    .groupby("incoming_record_id")["is_same_person"]
    .max()
    .mean()
)

recall_at_5_test = (
    scored_test[scored_test["rank"] <= 5]
    .groupby("incoming_record_id")["is_same_person"]
    .max()
    .mean()
)

print("\nML Top-1 accuracy on HELD-OUT TEST incoming:", ml_top1_test)
print("ML Recall@3 on HELD-OUT TEST incoming:", recall_at_3_test)
print("ML Recall@5 on HELD-OUT TEST incoming:", recall_at_5_test)
print("Incoming evaluated (TEST):", top1_test["incoming_record_id"].nunique())

# ----------------------------
# 6) Business baseline on same TEST incoming IDs
# ----------------------------
business_recall_test = None

if "true" in globals() and "incoming_record_id" in true.columns and "correct_match" in true.columns:
    tmp = true[true["incoming_record_id"].isin(test_ids_list)].copy()
    business_recall_test = float(tmp["correct_match"].mean())

elif "eval_df" in globals() and "incoming_record_id" in eval_df.columns:
    if "pred_db_record_id" in eval_df.columns and "db_record_id" in eval_df.columns:
        tmp = eval_df[eval_df["incoming_record_id"].isin(test_ids_list)].copy()
        business_recall_test = float((tmp["pred_db_record_id"] == tmp["db_record_id"]).mean())

if business_recall_test is not None:
    print("\nBusiness baseline match rate on HELD-OUT TEST incoming:", business_recall_test)
    print("Absolute lift vs business:", ml_top1_test - business_recall_test)
else:
    print("\nCould not compute business baseline automatically on TEST.")
    print("Please filter your business-logic evaluation table to test_ids and compare there.")

Held-out incoming records: 9951
Avg candidates per incoming (TEST): 155.60375841623957
Candidate recall on TEST: 1.0
Scored TEST rows 0..20,000
Scored TEST rows 20,000..40,000
Scored TEST rows 40,000..60,000
Scored TEST rows 60,000..80,000
Scored TEST rows 80,000..100,000
Scored TEST rows 100,000..120,000
Scored TEST rows 120,000..140,000
Scored TEST rows 140,000..160,000
Scored TEST rows 160,000..180,000
Scored TEST rows 180,000..200,000
Scored TEST rows 200,000..220,000
Scored TEST rows 220,000..240,000
Scored TEST rows 240,000..260,000
Scored TEST rows 260,000..280,000
Scored TEST rows 280,000..300,000
Scored TEST rows 300,000..320,000
Scored TEST rows 320,000..340,000
Scored TEST rows 340,000..360,000
Scored TEST rows 360,000..380,000
Scored TEST rows 380,000..400,000
Scored TEST rows 400,000..420,000
Scored TEST rows 420,000..440,000
Scored TEST rows 440,000..460,000
Scored TEST rows 460,000..480,000
Scored TEST rows 480,000..500,000
Scored TEST rows 500,000..520,000
Scored TEST r

# Permutation Sanity Check

To verify that the model's performance is driven by real signal rather than artifacts or leakage, run a permutation test.

In this test, the training labels are randomly shuffled so that the relationship between features and the true match label is destroyed. The model is then retrained and evaluated using the same pipeline and cross-validation procedure.

If the model's performance on the real data is due to meaningful structure in the features, performance should collapse to chance levels when the labels are randomized.

In [68]:
# --- Permutation sanity check: if labels are randomized, performance should collapse to chance ---
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def permuted_cv_auc_ap(seed, C=0.01):
    rng = np.random.default_rng(seed)
    y_perm = rng.permutation(y_train.values)

    fold_aucs, fold_aps = [], []

    for tr_idx, te_idx in cv.split(X_train, y_perm):
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=C, penalty="l1", solver="liblinear", max_iter=2000))
        ])

        pipe.fit(X_train.iloc[tr_idx], y_perm[tr_idx])
        proba = pipe.predict_proba(X_train.iloc[te_idx])[:, 1]

        fold_aucs.append(roc_auc_score(y_perm[te_idx], proba))
        fold_aps.append(average_precision_score(y_perm[te_idx], proba))

    return np.mean(fold_aucs), np.mean(fold_aps)

n_permutations = 30
cv_aucs, cv_aps = [], []

for seed in range(n_permutations):
    auc_mean, ap_mean = permuted_cv_auc_ap(seed, C=1.0)
    cv_aucs.append(auc_mean)
    cv_aps.append(ap_mean)

cv_aucs = np.array(cv_aucs)
cv_aps = np.array(cv_aps)

print(f"[CV permutation] AUC mean={cv_aucs.mean():.4f}, median={np.median(cv_aucs):.4f}")
print(f"[CV permutation] AP  mean={cv_aps.mean():.4f}, median={np.median(cv_aps):.4f}")
print(f"[CV permutation] AUC std={cv_aucs.std():.4f}")
print(f"[CV permutation] AP  std={cv_aps.std():.4f}")

[CV permutation] AUC mean=0.4993, median=0.4992
[CV permutation] AP  mean=0.0196, median=0.0196
[CV permutation] AUC std=0.0023
[CV permutation] AP  std=0.0002


### Interpretation

Under label permutation, the model's performance collapses to chance levels. The mean cross-validated AUC is 0.5005, which is effectively random ranking, and the mean Average Precision is 0.0197, which is approximately equal to the positive class prevalence.

The very small standard deviations across permutations (AUC std = 0.0018, AP std = 0.0001) indicate that this chance-level behavior is stable and consistent rather than the result of a particular random shuffle.

This provides strong evidence that the model's performance with the true labels is driven by real signal in the engineered features rather than random chance, label leakage, or artifacts in the evaluation pipeline.

# ERROR ANALYSIS

In [71]:
# --- 1) Tag which incoming records had the truth in the candidate set ---
gt = truth[truth["is_true_match"] == 1][["incoming_record_id", "db_record_id"]].copy()
gt = gt.rename(columns={"db_record_id": "true_db_record_id"})

# Did blocking include the truth?
cand_truth_hit = candidates_pruned.merge(
    gt,
    left_on=["incoming_record_id", "db_record_id"],
    right_on=["incoming_record_id", "true_db_record_id"],
    how="inner"
)[["incoming_record_id"]].drop_duplicates()
cand_truth_hit["truth_in_candidates"] = 1

incoming_flag = gt.merge(cand_truth_hit, on="incoming_record_id", how="left")
incoming_flag["truth_in_candidates"] = incoming_flag["truth_in_candidates"].fillna(0).astype(int)

print("Candidate recall:", incoming_flag["truth_in_candidates"].mean())
print("Blocking misses:", int((incoming_flag["truth_in_candidates"] == 0).sum()), "/", len(incoming_flag))

# --- 2) Top-1 prediction per incoming (reranker output) ---
# pairs_scored must include incoming_record_id, db_record_id, proba
top1 = (scored_all
        .sort_values(["incoming_record_id", "proba"], ascending=[True, False])
        .groupby("incoming_record_id", as_index=False)
        .head(1)
        .rename(columns={"db_record_id": "pred_db_record_id", "proba": "top1_proba"})
       )

# Also capture runner-up for margin diagnostics (optional but very useful)
top2 = (scored_all
        .sort_values(["incoming_record_id", "proba"], ascending=[True, False])
        .groupby("incoming_record_id", as_index=False)
        .head(2)
       )
ranked = top2.groupby("incoming_record_id")["proba"].apply(lambda s: list(s.values)).reset_index(name="top2_list")
ranked["top1_proba"] = ranked["top2_list"].apply(lambda x: x[0] if len(x) > 0 else np.nan)
ranked["top2_proba"] = ranked["top2_list"].apply(lambda x: x[1] if len(x) > 1 else np.nan)
ranked["top1_margin"] = ranked["top1_proba"] - ranked["top2_proba"]

top1 = top1.merge(ranked[["incoming_record_id","top2_proba","top1_margin"]], on="incoming_record_id", how="left")

# --- 3) Build error table and filter to ML-only errors ---
eval_top1 = (incoming_flag
             .merge(top1, on="incoming_record_id", how="left")
            )

eval_top1["ml_correct"] = (eval_top1["pred_db_record_id"] == eval_top1["true_db_record_id"]).astype(int)

# ML-only errors: truth was in candidates AND ML picked wrong
ml_errors = eval_top1[(eval_top1["truth_in_candidates"] == 1) & (eval_top1["ml_correct"] == 0)].copy()

print("\nML errors (excluding blocking misses):", len(ml_errors), "/", int((incoming_flag["truth_in_candidates"]==1).sum()))
print("Top-1 accuracy conditional on truth-in-candidates:",
      1 - (len(ml_errors) / max(1, int((incoming_flag["truth_in_candidates"]==1).sum()))))

ml_errors.head()

Candidate recall: 0.99508
Blocking misses: 246 / 50000

ML errors (excluding blocking misses): 634 / 49754
Top-1 accuracy conditional on truth-in-candidates: 0.9872573059452506


,incoming_record_id,true_db_record_id,truth_in_candidates,pred_db_record_id,is_same_person,top1_proba,top2_proba,top1_margin,ml_correct
54,55,20149,1,19686.0,0,0.814558,0.748554,0.066004,0
79,80,126374,1,37845.0,0,0.794067,0.794067,0.000000,0
89,90,177248,1,118963.0,0,0.789644,0.771555,0.018089,0
106,107,150861,1,53629.0,0,0.812852,0.812852,0.000000,0
126,127,84724,1,142902.0,0,0.826977,0.793979,0.032998,0


In [72]:
# Bring in details from pairs_scored for both the predicted and true candidates
pairs_cols = ["incoming_record_id","db_record_id","proba"]  # add left/right raw columns if present
pairs_base = scored_all.copy()

# True candidate score
true_rows = pairs_base.merge(
    ml_errors[["incoming_record_id","true_db_record_id"]],
    left_on=["incoming_record_id","db_record_id"],
    right_on=["incoming_record_id","true_db_record_id"],
    how="inner"
).rename(columns={"proba":"true_proba"}).drop(columns=["true_db_record_id"])

# Pred candidate score (top1)
pred_rows = pairs_base.merge(
    ml_errors[["incoming_record_id","pred_db_record_id"]],
    left_on=["incoming_record_id","db_record_id"],
    right_on=["incoming_record_id","pred_db_record_id"],
    how="inner"
).rename(columns={"proba":"pred_proba"}).drop(columns=["pred_db_record_id"])

# Combine
err_detail = (ml_errors
              .merge(true_rows[["incoming_record_id","db_record_id","true_proba"]], on=["incoming_record_id"], how="left")
              .merge(pred_rows[["incoming_record_id","db_record_id","pred_proba"]], on=["incoming_record_id"], how="left", suffixes=("_true","_pred"))
             )

err_detail["proba_gap_true_minus_pred"] = err_detail["true_proba"] - err_detail["pred_proba"]

# Sort by "most painful" mistakes (true much higher than pred shouldn't happen; near-ties are normal)
err_detail.sort_values("proba_gap_true_minus_pred").head()

,incoming_record_id,true_db_record_id,truth_in_candidates,pred_db_record_id,is_same_person,top1_proba,top2_proba,top1_margin,ml_correct,db_record_id_true,true_proba,db_record_id_pred,pred_proba,proba_gap_true_minus_pred
321,25364,99527,1,4250.0,0,0.769360,0.400367,0.368993,0,99527.0,0.400367,4250.0,0.769360,-0.368993
278,22133,162616,1,35001.0,0,0.763531,0.400367,0.363164,0,162616.0,0.400367,35001.0,0.763531,-0.363164
25,1394,66191,1,196601.0,0,0.759805,0.400367,0.359438,0,66191.0,0.400367,196601.0,0.759805,-0.359438
465,36491,186394,1,69002.0,0,0.755863,0.400367,0.355496,0,186394.0,0.400367,69002.0,0.755863,-0.355496
112,8060,83939,1,117993.0,0,0.748424,0.415087,0.333337,0,83939.0,0.415087,117993.0,0.748424,-0.333337


In [73]:
import pandas as pd
import numpy as np
import re
from dateutil import parser

# -----------------------------
# ML-only error review export
# -----------------------------

# 1) Ensure top1 has the expected names
top1_fix = top1.copy()
if "pred_db_record_id" not in top1_fix.columns and "db_record_id" in top1_fix.columns:
    top1_fix = top1_fix.rename(columns={"db_record_id": "pred_db_record_id"})
if "top1_proba" not in top1_fix.columns and "proba" in top1_fix.columns:
    top1_fix = top1_fix.rename(columns={"proba": "top1_proba"})

# 2) Build base error frame with prediction metadata
errors = ml_errors.merge(
    top1_fix[["incoming_record_id","pred_db_record_id","top1_proba","top2_proba","top1_margin"]],
    on="incoming_record_id", how="left"
).merge(
    err_detail[["incoming_record_id","true_proba","pred_proba","proba_gap_true_minus_pred"]],
    on="incoming_record_id", how="left"
)

# 3) Choose which raw columns to export (edit if you want more)
INC_COLS = ["incoming_record_id","indexor_code","patient_name","patient_ssn","claim_id","patient_dob","patient_doi"]
DB_COLS  = ["db_record_id","indexor_code","patient_name","patient_ssn","claim_id","patient_dob","patient_doi"]

inc_small = inc[INC_COLS].copy()
db_small  = db[DB_COLS].copy()

# 4) Merge incoming details
out = errors.merge(inc_small, on="incoming_record_id", how="left")

# 5) Merge TRUE DB details
true_db = db_small.rename(columns={c: f"true_{c}" for c in db_small.columns if c != "db_record_id"})
true_db = true_db.rename(columns={"db_record_id": "true_db_record_id"})
out = out.merge(true_db, on="true_db_record_id", how="left")

# 6) Ensure pred_db_record_id exists before merging predicted DB details (prevents KeyError)
if "pred_db_record_id" not in out.columns:
    out = out.merge(top1_fix[["incoming_record_id","pred_db_record_id"]], on="incoming_record_id", how="left")

# 7) Merge PRED DB details
pred_db = db_small.rename(columns={c: f"pred_{c}" for c in db_small.columns if c != "db_record_id"})
pred_db = pred_db.rename(columns={"db_record_id": "pred_db_record_id"})
out = out.merge(pred_db, on="pred_db_record_id", how="left")

# 8) Add quick diffs for easier eyeballing
def parse_date_safe(x):
    if pd.isna(x) or x in ("", "N/A", "NA", None):
        return pd.NaT
    try:
        return parser.parse(str(x), fuzzy=True, dayfirst=False).date()
    except Exception:
        return pd.NaT

def norm_digits(x):
    if pd.isna(x): return ""
    return "".join(ch for ch in str(x) if ch.isalnum()).lower()

def norm_name(x):
    if pd.isna(x) or not isinstance(x, str): return ""
    s = x.strip().lower().replace("'", " ").replace("-", " ")
    s = "".join(ch if ch.isalnum() or ch.isspace() else " " for ch in s)
    return " ".join(s.split())

out["inc_dob_dt"]  = out["patient_dob"].apply(parse_date_safe)
out["true_dob_dt"] = out["true_patient_dob"].apply(parse_date_safe)
out["pred_dob_dt"] = out["pred_patient_dob"].apply(parse_date_safe)

out["inc_doi_dt"]  = out["patient_doi"].apply(parse_date_safe)
out["true_doi_dt"] = out["true_patient_doi"].apply(parse_date_safe)
out["pred_doi_dt"] = out["pred_patient_doi"].apply(parse_date_safe)

out["dob_match_true"] = (out["inc_dob_dt"] == out["true_dob_dt"])
out["dob_match_pred"] = (out["inc_dob_dt"] == out["pred_dob_dt"])

out["doi_diff_true_days"] = (pd.to_datetime(out["inc_doi_dt"]) - pd.to_datetime(out["true_doi_dt"])).dt.days.abs()
out["doi_diff_pred_days"] = (pd.to_datetime(out["inc_doi_dt"]) - pd.to_datetime(out["pred_doi_dt"])).dt.days.abs()

out["true_ssn_match"]   = (out["patient_ssn"].apply(norm_digits) == out["true_patient_ssn"].apply(norm_digits))
out["pred_ssn_match"]   = (out["patient_ssn"].apply(norm_digits) == out["pred_patient_ssn"].apply(norm_digits))
out["true_claim_match"] = (out["claim_id"].apply(norm_digits) == out["true_claim_id"].apply(norm_digits))
out["pred_claim_match"] = (out["claim_id"].apply(norm_digits) == out["pred_claim_id"].apply(norm_digits))

out["true_name_match_norm"] = (out["patient_name"].apply(norm_name) == out["true_patient_name"].apply(norm_name))
out["pred_name_match_norm"] = (out["patient_name"].apply(norm_name) == out["pred_patient_name"].apply(norm_name))

# 9) Reorder columns for readability
front = [
    "incoming_record_id","true_db_record_id","pred_db_record_id",
    "top1_proba","top2_proba","top1_margin",
    "true_proba","pred_proba","proba_gap_true_minus_pred",
    "indexor_code","patient_name","patient_ssn","claim_id","patient_dob","patient_doi",
    "true_patient_name","true_patient_ssn","true_claim_id","true_patient_dob","true_patient_doi",
    "pred_patient_name","pred_patient_ssn","pred_claim_id","pred_patient_dob","pred_patient_doi",
    "dob_match_true","dob_match_pred","doi_diff_true_days","doi_diff_pred_days",
    "true_ssn_match","pred_ssn_match","true_claim_match","pred_claim_match",
    "true_name_match_norm","pred_name_match_norm"
]
front = [c for c in front if c in out.columns]
out = out[front + [c for c in out.columns if c not in front]]

# 10) Export
out_path = "ml_only_errors_review.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    out.to_excel(writer, index=False, sheet_name="ml_only_errors")

    summary = pd.DataFrame({
        "metric": [
            "total_true_incoming",
            "candidate_recall",
            "blocking_misses",
            "ml_only_errors",
            "top1_acc_cond_truth_in_candidates"
        ],
        "value": [
            len(incoming_flag),
            float(incoming_flag["truth_in_candidates"].mean()),
            int((incoming_flag["truth_in_candidates"] == 0).sum()),
            len(ml_errors),
            1 - (len(ml_errors) / max(1, int((incoming_flag["truth_in_candidates"] == 1).sum())))
        ]
    })
    summary.to_excel(writer, index=False, sheet_name="summary")

print("Wrote:", out_path)
print("Rows exported:", len(out))

Wrote: ml_only_errors_review.xlsx
Rows exported: 634
